# Document-Grounded Q&A System for Long Documents

**Goal:** Answer questions accurately from a single long document (100+ pages) with citations and hallucination resistance.

**Pipeline:**
1. **Ingestion** — Parse PDF, extract text + metadata per page
2. **Chunking** — Split into overlapping, section-aware chunks
3. **Indexing** — FAISS (semantic) + BM25 (keyword) dual index
4. **Retrieval** — Hybrid search with Reciprocal Rank Fusion
5. **Generation** — LLM with strict grounding prompt + citations
6. **Evaluation** — LLM-as-judge scoring

---
## 1. Setup

In [ ]:
!pip install pymupdf sentence-transformers faiss-cpu rank_bm25 openai tiktoken -q

In [2]:
import os
import re
import json
import hashlib
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Tuple, Any
from pathlib import Path
import warnings
from dotenv import load_dotenv
warnings.filterwarnings("ignore")

import fitz  # PyMuPDF
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from openai import OpenAI
import tiktoken

load_dotenv()
print("All imports successful.")

All imports successful.


In [3]:
# ── Configuration ─────────────────────────────────────────
# All tunable parameters in one place.

CONFIG = {
    # Paths
    "pdf_path": "document.pdf",

    # Chunking
    "chunk_size": 512,          # tokens per chunk
    "chunk_overlap": 64,        # overlap tokens between chunks

    # Embedding
    "embedding_model": "BAAI/bge-base-en-v1.5",

    # Retrieval
    "top_k": 15,                # candidates from each retriever
    "top_k_final": 6,           # passages sent to LLM
    "rrf_k": 60,                # RRF smoothing constant

    # Generation
    "llm_model": "gpt-4o-mini",
    "temperature": 0.0,
    "max_tokens": 1024,
}

# API key
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    print("OPENAI_API_KEY not set. Run: os.environ['OPENAI_API_KEY'] = 'sk-...'")
else:
    print("OpenAI API key loaded.")

print(f"Embedding: {CONFIG['embedding_model']}")
print(f"Chunks: {CONFIG['chunk_size']} tokens, {CONFIG['chunk_overlap']} overlap")
print(f"LLM: {CONFIG['llm_model']}")

OpenAI API key loaded.
Embedding: BAAI/bge-base-en-v1.5
Chunks: 512 tokens, 64 overlap
LLM: gpt-4o-mini


---
## 2. Data Structures

Simple dataclasses to hold extracted content and chunks.

In [4]:
@dataclass
class PageElement:
    """A block of text extracted from one PDF page."""
    text: str
    page_num: int               # 1-indexed
    element_type: str           # 'heading', 'paragraph', 'table'
    section_title: str = ""     # nearest heading above this element


@dataclass
class Chunk:
    """A chunk of text ready for indexing and retrieval."""
    chunk_id: str
    text: str
    page_numbers: List[int]
    section_title: str
    token_count: int
    chunk_index: int            # position in document order

    def citation(self) -> str:
        pages = sorted(set(self.page_numbers))
        if len(pages) == 1:
            p = f"p. {pages[0]}"
        elif len(pages) <= 3:
            p = f"pp. {', '.join(map(str, pages))}"
        else:
            p = f"pp. {pages[0]}-{pages[-1]}"
        sec = f' — "{self.section_title}"' if self.section_title else ""
        return f"[{p}{sec}]"

print("Data structures defined.")

Data structures defined.


---
## 3. PDF Ingestion

**Why PyMuPDF?** Fast, good structure extraction, handles most layouts.

We extract text block-by-block from each page. Headings are detected using
a simple heuristic: blocks with font size significantly above the page median
are likely headings.

In [5]:
def get_median_font_size(page) -> float:
    """Compute the median font size on a page for heading detection."""
    sizes = []
    blocks = page.get_text("dict", flags=fitz.TEXT_PRESERVE_WHITESPACE)["blocks"]
    for b in blocks:
        if b.get("type", 1) != 0:  # skip non-text (image) blocks
            continue
        for line in b.get("lines", []):
            for span in line.get("spans", []):
                t = span.get("text", "").strip()
                if len(t) > 2:
                    sizes.extend([span["size"]] * len(t))
    return float(np.median(sizes)) if sizes else 12.0


def detect_heading(block: dict, median_size: float) -> Tuple[bool, int]:
    """
    Heuristic: if the block's font is notably larger than the page median,
    and the text is short, it's probably a heading.
    Returns (is_heading, level).
    """
    lines = block.get("lines", [])
    if not lines:
        return False, 0

    # Get dominant font size and check for bold
    font_sizes = []
    is_bold = False
    for line in lines:
        for span in line.get("spans", []):
            font_sizes.append(span["size"])
            if "bold" in span.get("font", "").lower():
                is_bold = True

    if not font_sizes:
        return False, 0

    block_size = max(font_sizes)
    ratio = block_size / median_size if median_size > 0 else 1.0

    # Reconstruct the block's full text
    full_text = " ".join(
        span["text"]
        for line in lines
        for span in line.get("spans", [])
    ).strip()

    # Too long to be a heading
    if len(full_text) > 200 or len(full_text) < 2:
        return False, 0

    if ratio >= 1.5 or (ratio >= 1.3 and is_bold):
        return True, 1
    elif ratio >= 1.2 or (is_bold and len(full_text) < 80):
        return True, 2

    return False, 0


def extract_block_text(block: dict) -> str:
    """Get all text from a block's spans."""
    text = ""
    for line in block.get("lines", []):
        line_text = "".join(span["text"] for span in line.get("spans", []))
        text += line_text + "\n"
    return text.strip()


def ingest_pdf(pdf_path: str) -> Tuple[List[PageElement], dict]:
    """
    Parse a PDF and return structured elements with metadata.

    Returns:
        elements: list of PageElement
        metadata: dict with title, pages, etc.
    """
    doc = fitz.open(pdf_path)
    elements = []
    current_section = ""

    metadata = {
        "title": doc.metadata.get("title", ""),
        "author": doc.metadata.get("author", ""),
        "total_pages": len(doc),
        "file_name": os.path.basename(pdf_path),
    }

    for page_idx in range(len(doc)):
        page = doc[page_idx]
        page_num = page_idx + 1
        median_fs = get_median_font_size(page)

        blocks = page.get_text("dict", flags=fitz.TEXT_PRESERVE_WHITESPACE)["blocks"]

        # Also try to extract tables
        try:
            tables = page.find_tables()
            for table in tables:
                df = table.to_pandas()
                if not df.empty:
                    elements.append(PageElement(
                        text=df.to_markdown(index=False),
                        page_num=page_num,
                        element_type="table",
                        section_title=current_section,
                    ))
        except Exception:
            pass  # Table extraction can fail; that's ok for the initial implementation

        for block in blocks:
            if block.get("type", 1) != 0:  # skip image blocks
                continue

            text = extract_block_text(block)
            if not text or len(text) < 3:
                continue

            is_heading, level = detect_heading(block, median_fs)

            if is_heading:
                current_section = text.strip()
                elements.append(PageElement(
                    text=text,
                    page_num=page_num,
                    element_type="heading",
                    section_title=current_section,
                ))
            else:
                elements.append(PageElement(
                    text=text,
                    page_num=page_num,
                    element_type="paragraph",
                    section_title=current_section,
                ))

    doc.close()

    print(f"Ingested {metadata['total_pages']} pages → {len(elements)} elements")
    types = {}
    for e in elements:
        types[e.element_type] = types.get(e.element_type, 0) + 1
    for t, c in sorted(types.items()):
        print(f"   {t}: {c}")

    return elements, metadata

print("Ingestion functions defined.")

Ingestion functions defined.


In [6]:
# ── Run Ingestion ─────────────────────────────────────────
PDF_PATH = "document.pdf"  # ← CHANGE THIS to your PDF path

if not os.path.exists(PDF_PATH):
    print(f"File not found: {PDF_PATH}")
    print("Set PDF_PATH to your document.")
else:
    elements, doc_metadata = ingest_pdf(PDF_PATH)
    print(f"\n Document: {doc_metadata.get('title') or doc_metadata['file_name']}")
    print(f"   Pages: {doc_metadata['total_pages']}")
    print(f"\n--- Sample Elements ---")
    for e in elements[:5]:
        preview = e.text[:120].replace('\n', ' ')
        print(f"  [p{e.page_num}] ({e.element_type}) {preview}...")

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Ingested 160 pages → 905 elements
   heading: 170
   paragraph: 730
   table: 5

 Document: 10-K - 02/07/2019 - 3M Company
   Pages: 160

--- Sample Elements ---
  [p1] (paragraph) low...
  [p1] (heading) UNITED STATES SECURITIES AND EXCHANGE COMMISSION...
  [p1] (heading) Washington, D.C. 20549...
  [p1] (heading) FORM 10-K...
  [p1] (heading) ☒   ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE...


---
## 4. Chunking

**Strategy:** Group consecutive elements by section heading, then split into
token-bounded chunks with overlap. This keeps topically related content together.

Why token-based (not character-based): tokens map directly to what the LLM sees,
so a 512-token chunk uses a predictable amount of context window.

In [10]:
tokenizer = tiktoken.encoding_for_model("gpt-4o-mini")

def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text))


def chunk_elements(elements: List[PageElement], chunk_size=512, overlap=64) -> List[Chunk]:
    """
    Convert elements into overlapping, section-aware chunks.

    1. Group by section heading
    2. Accumulate text until chunk_size is reached
    3. Overlap by keeping trailing tokens from the previous chunk
    """
    chunks = []
    chunk_idx = 0

    # Group elements by section
    sections = []
    current_title = ""
    current_elems = []
    for elem in elements:
        if elem.element_type == "heading":
            if current_elems:
                sections.append((current_title, current_elems))
            current_title = elem.text.strip()
            current_elems = [elem]
        else:
            current_elems.append(elem)
    if current_elems:
        sections.append((current_title, current_elems))

    # Chunk each section
    for section_title, section_elems in sections:
        buf_texts = []
        buf_pages = []
        buf_tokens = 0

        for elem in section_elems:
            elem_tokens = count_tokens(elem.text)

            # If single element exceeds chunk_size, split it
            if elem_tokens > chunk_size:
                # Flush buffer first
                if buf_texts:
                    chunks.append(_make_chunk(buf_texts, buf_pages, section_title, chunk_idx))
                    chunk_idx += 1
                    buf_texts, buf_pages, buf_tokens = [], [], 0

                # Split large element by tokens
                tokens = tokenizer.encode(elem.text)
                for i in range(0, len(tokens), chunk_size):
                    piece = tokenizer.decode(tokens[i:i + chunk_size])
                    chunks.append(_make_chunk([piece], [elem.page_num], section_title, chunk_idx))
                    chunk_idx += 1
                continue

            # Would this exceed chunk_size? If so, flush.
            if buf_tokens + elem_tokens > chunk_size and buf_texts:
                chunks.append(_make_chunk(buf_texts, buf_pages, section_title, chunk_idx))
                chunk_idx += 1

                # Overlap: keep trailing content that fits in overlap budget
                keep_texts, keep_pages, keep_tokens = [], [], 0
                for t, p in zip(reversed(buf_texts), reversed(buf_pages)):
                    t_tok = count_tokens(t)
                    if keep_tokens + t_tok > overlap:
                        break
                    keep_texts.insert(0, t)
                    keep_pages.insert(0, p)
                    keep_tokens += t_tok
                buf_texts, buf_pages, buf_tokens = keep_texts, keep_pages, keep_tokens

            buf_texts.append(elem.text)
            buf_pages.append(elem.page_num)
            buf_tokens += elem_tokens

        # Flush remaining
        if buf_texts:
            chunks.append(_make_chunk(buf_texts, buf_pages, section_title, chunk_idx))
            chunk_idx += 1

    # Filter tiny chunks
    chunks = [c for c in chunks if c.token_count >= 30]

    print(f"✅ {len(chunks)} chunks created")
    toks = [c.token_count for c in chunks]
    print(f"   Tokens: min={min(toks)}, max={max(toks)}, avg={np.mean(toks):.0f}")
    return chunks


def _make_chunk(texts, pages, section_title, idx):
    combined = "\n\n".join(texts)
    cid = hashlib.md5(f"{idx}:{combined[:50]}".encode()).hexdigest()[:10]
    return Chunk(
        chunk_id=cid,
        text=combined,
        page_numbers=list(pages),
        section_title=section_title,
        token_count=count_tokens(combined),
        chunk_index=idx,
    )

print("Chunking functions defined.")

Chunking functions defined.


In [8]:
tokenizer.encode("hello world")

[24912, 2375]

In [9]:
tokenizer.decode([24912, 2375])

'hello world'

In [12]:
# ── Run Chunking ──────────────────────────────────────────
if 'elements' in dir():
    chunks = chunk_elements(
        elements,
        chunk_size=CONFIG["chunk_size"],
        overlap=CONFIG["chunk_overlap"],
    )
    print(f"\n--- Sample Chunks ---")
    for c in chunks[:3]:
        print(f"  [{c.chunk_index}] {c.citation()} ({c.token_count} tok)")
        print(f"    {c.text[:120].replace(chr(10), ' ')}...")
        print()
else:
    print("⚠️  Run ingestion first.")

✅ 368 chunks created
   Tokens: min=33, max=561, avg=379

--- Sample Chunks ---
  [7] [p. 1 — "3M COMPANY
State of Incorporation: Delaware
 
I.R.S. Employer Identification No. 41-0417775
Principal executive offices: 3M Center, St. Paul, Minnesota 55144"] (43 tok)
    3M COMPANY State of Incorporation: Delaware   I.R.S. Employer Identification No. 41-0417775 Principal executive offices:...

  [8] [p. 1 — "Telephone number: (651) 733-1110"] (36 tok)
    Telephone number: (651) 733-1110  SECURITIES REGISTERED PURSUANT TO SECTION 12(b) OF THE ACT:       Title of each class...

  [10] [p. 1 — "on which registered
Common Stock, Par Value $.01 Per Share"] (471 tok)
    on which registered Common Stock, Par Value $.01 Per Share  1.500% Notes due 2026 Floating Rate Notes due 2020  0.375% N...



---
## 5. Hybrid Index & Retrieval

We build two complementary indices:

| Method | Strength | Weakness |
|--------|----------|----------|
| **FAISS** (dense embeddings) | Semantic similarity, paraphrases | Can miss exact terms |
| **BM25** (keyword matching) | Exact terms, numbers, acronyms | No semantic understanding |

**Reciprocal Rank Fusion (RRF)** merges their rankings without needing score normalization:

`RRF(doc) = Σ 1 / (k + rank_i(doc))`

In [13]:
class HybridIndex:
    """FAISS + BM25 hybrid index with RRF fusion."""

    def __init__(self, embedding_model_name: str):
        print(f"Loading embedding model: {embedding_model_name}...")
        self.embed_model = SentenceTransformer(embedding_model_name)
        self.chunks: List[Chunk] = []
        self.faiss_index = None
        self.bm25 = None
        print("✅ Embedding model loaded.")

    def build(self, chunks: List[Chunk]):
        self.chunks = chunks
        texts = [c.text for c in chunks]

        # Dense index
        print("Encoding chunks...")
        embeddings = self.embed_model.encode(
            texts, show_progress_bar=True, batch_size=32, normalize_embeddings=True
        )
        dim = embeddings.shape[1]
        self.faiss_index = faiss.IndexFlatIP(dim)  # inner product = cosine for normalized
        self.faiss_index.add(embeddings.astype(np.float32))
        print(f"  FAISS: {self.faiss_index.ntotal} vectors, {dim}d")

        # Sparse index
        tokenized = [self._tokenize(t) for t in texts]
        self.bm25 = BM25Okapi(tokenized)
        print(f"  BM25: {len(tokenized)} documents")
        print("✅ Index built.")

    @staticmethod
    def _tokenize(text: str) -> List[str]:
        text = re.sub(r'[^a-z0-9\s]', ' ', text.lower())
        return [t for t in text.split() if len(t) > 1]

    def search(self, query: str, top_k: int = 15, rrf_k: int = 60) -> List[Tuple[Chunk, float]]:
        """Hybrid search: FAISS + BM25 combined with RRF."""

        # Dense search
        q_emb = self.embed_model.encode(
            [query], normalize_embeddings=True
        ).astype(np.float32)
        scores_d, indices_d = self.faiss_index.search(q_emb, top_k)
        dense_ranking = [(int(idx), rank) for rank, idx in enumerate(indices_d[0]) if idx >= 0]

        # Sparse search
        q_tokens = self._tokenize(query)
        bm25_scores = self.bm25.get_scores(q_tokens)
        top_sparse = np.argsort(bm25_scores)[::-1][:top_k]
        sparse_ranking = [(int(idx), rank) for rank, idx in enumerate(top_sparse) if bm25_scores[idx] > 0]

        # RRF fusion
        rrf = {}
        for idx, rank in dense_ranking:
            rrf[idx] = rrf.get(idx, 0) + 1.0 / (rrf_k + rank + 1)
        for idx, rank in sparse_ranking:
            rrf[idx] = rrf.get(idx, 0) + 1.0 / (rrf_k + rank + 1)

        sorted_results = sorted(rrf.items(), key=lambda x: x[1], reverse=True)
        return [(self.chunks[idx], score) for idx, score in sorted_results[:top_k]]

print("HybridIndex defined.")

HybridIndex defined.


In [14]:
# ── Build Index ───────────────────────────────────────────
if 'chunks' in dir():
    index = HybridIndex(CONFIG["embedding_model"])
    index.build(chunks)
else:
    print("⚠️  Run chunking first.")

Loading embedding model: BAAI/bge-base-en-v1.5...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6969.03it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded.
Encoding chunks...


Batches: 100%|██████████| 12/12 [00:04<00:00,  2.98it/s]

  FAISS: 368 vectors, 768d
  BM25: 368 documents
✅ Index built.


In [22]:
# ── Test Retrieval ────────────────────────────────────────
if 'index' in dir():
    test_q = "What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement."  # ← adjust for your document
    print(f"Query: '{test_q}'\n")
    results = index.search(test_q, top_k=5)
    for i, (chunk, score) in enumerate(results, 1):
        print(f"  {i}. [{score:.4f}] {chunk.citation()}")
        print(f"     {chunk.text[:100].replace(chr(10), ' ')}...")
        print()

Query: 'What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.'

  1. [0.0164] [p. 49 — "U.S.
Postretirement"]
     . Refer to the preceding “Cash Flows from Investing Activities” section for discussion on capital sp...

  2. [0.0164] [p. 49 — "U.S.
Postretirement"]
     Other cash flows from financing activities may include various other items, such as changes in cash ...

  3. [0.0161] [p. 44 — "U.S.
Postretirement"]
     Table of Contents  Cash, Cash Equivalents and Marketable Securities:   At December 31, 2018, 3M had ...

  4. [0.0161] [p. 26 — "Resolution
  
 —   
897  
 —   
897   
127  
   
770   
1.28  
  
Full Year 2018 Adjusted"]
     Foreign currency translation increased year-on-year sales by 0.5 percent, with the translation-relat...

  5. [0.0159] [p. 47 — "U.S.
Postretirement"]
     Investments in property, plant and equipment enable growth across many diverse ma

---
## 6. Answer Generation

The prompt is designed to prevent hallucination:
- Only use provided context
- Cite sources with `[Source N]` notation
- If evidence is insufficient, say so explicitly
- Don't infer beyond what's stated

In [24]:
SYSTEM_PROMPT = """You are a precise document analysis assistant. Answer questions based
ONLY on the provided source passages.

RULES:
1. Use ONLY information from the provided sources. No external knowledge.
2. Cite sources using [Source N] notation.
3. If multiple sources support a claim, cite all: [Source 1, Source 3].
4. If the sources don't contain enough information, say:
   "The provided document sections do not contain sufficient information to answer this question."
5. Do not speculate beyond what the sources explicitly state.
6. For numbers, quote exact figures from the sources.
7. Be concise but complete."""


def generate_answer(query: str, passages: List[Tuple[Chunk, float]], config: dict) -> dict:
    """Generate a grounded answer with citations."""
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))

    # Format context
    context_parts = []
    for i, (chunk, score) in enumerate(passages, 1):
        context_parts.append(f"[Source {i}] {chunk.citation()}\n{chunk.text}")
    context = "\n\n---\n\n".join(context_parts)

    user_msg = f"""Based on the following source passages, answer the question.

SOURCE PASSAGES:
{context}

QUESTION: {query}

Provide a precise, well-cited answer using only the sources above."""

    response = client.chat.completions.create(
        model=config["llm_model"],
        temperature=config["temperature"],
        max_tokens=config["max_tokens"],
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
    )

    answer = response.choices[0].message.content

    return {
        "query": query,
        "answer": answer,
        "sources": [
            {"source_num": i + 1, "chunk": chunk, "score": score}
            for i, (chunk, score) in enumerate(passages)
        ],
        "usage": {
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
        },
    }

print("Generation function defined.")

Generation function defined.


---
## 7. Full Pipeline

One function that ties retrieval → generation together.

In [25]:
def ask(query: str, index: HybridIndex, config: dict, verbose=True) -> dict:
    """
    Full Q&A pipeline: retrieve → generate → cite.
    """
    if verbose:
        print(f"\n{'─'*60}")
        print(f"Q: {query}")
        print(f"{'─'*60}")

    # Retrieve
    candidates = index.search(query, top_k=config["top_k"], rrf_k=config["rrf_k"])
    passages = candidates[:config["top_k_final"]]

    if verbose:
        print(f"📚 Retrieved {len(candidates)} candidates, using top {len(passages)}")

    # Generate
    result = generate_answer(query, passages, config)

    if verbose:
        print(f"\n💡 ANSWER:\n{result['answer']}")
        print(f"\n📖 Sources:")
        for s in result["sources"]:
            c = s["chunk"]
            print(f"   [Source {s['source_num']}] {c.citation()}")
            print(f"      {c.text[:80].replace(chr(10), ' ')}...")
        print(f"\n📊 Tokens used: {result['usage']['prompt_tokens']} prompt, "
              f"{result['usage']['completion_tokens']} completion")

    return result

print("Pipeline ready.")

Pipeline ready.


In [31]:
# ── Ask Questions ─────────────────────────────────────────
# Adjust these questions for YOUR document.

if 'index' in dir() and OPENAI_API_KEY:
    questions = [
        "What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.",
        "What is the correct religion?"  # ← This is a control question that should trigger the "insufficient information" response if the document doesn't contain religious information.
    ]
    results = []
    for q in questions:
        r = ask(q, index, CONFIG)
        results.append(r)
        print()
else:
    if 'index' not in dir():
        print("⚠️  Build the index first.")
    if not OPENAI_API_KEY:
        print("⚠️  Set OPENAI_API_KEY.")


────────────────────────────────────────────────────────────
Q: What is the FY2018 capital expenditure amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.
────────────────────────────────────────────────────────────
📚 Retrieved 15 candidates, using top 6

💡 ANSWER:
The FY2018 capital expenditure amount for 3M is $1,577 million, as indicated by the purchases of property, plant, and equipment (PP&E) in the cash flow statement [Source 4].

📖 Sources:
   [Source 1] [p. 50 — "Net income attributable to 3M
 
$
5,349  
$
4,858  
$
5,050"]
       of less than one year. Many of these commitments relate to take or pay contract...
   [Source 2] [p. 44 — "U.S.
Postretirement"]
      Table of Contents  Cash, Cash Equivalents and Marketable Securities:   At Decemb...
   [Source 3] [p. 43 — "U.S.
Postretirement"]
      FINANCIAL CONDITION AND LIQUIDI TY   The strength and stability of 3M’s business...
   [Source 4] [p. 49 — "U.

---
## 8. Evaluation

**LLM-as-Judge**: We use a separate LLM call to score answers on three dimensions:
- **Faithfulness** — Is everything in the answer supported by the sources?
- **Relevance** — Does it actually answer the question?
- **Citation Quality** — Are citations present and correct?

This isn't a replacement for human evaluation, but it provides a scalable signal
and shows the interviewer you're thinking about how to measure quality.

In [32]:
EVAL_PROMPT = """You are evaluating a document Q&A system. Given the question, answer,
and source passages, rate the answer on three dimensions (1-5 each).

QUESTION: {question}

SOURCES:
{sources}

ANSWER:
{answer}

Rate:
1. FAITHFULNESS (1-5): Is the answer fully supported by the sources?
2. RELEVANCE (1-5): Does it address the question?
3. CITATION_QUALITY (1-5): Are citations present, correct, sufficient?

Respond ONLY with JSON:
{{"faithfulness": <1-5>, "relevance": <1-5>, "citation_quality": <1-5>, "reasoning": "<brief explanation>"}}"""


def evaluate_result(result: dict) -> dict:
    """Score a single QA result using LLM-as-judge."""
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))

    sources_text = ""
    for s in result["sources"]:
        c = s["chunk"]
        sources_text += f"[Source {s['source_num']}] {c.citation()}\n{c.text[:400]}\n\n"

    prompt = EVAL_PROMPT.format(
        question=result["query"],
        sources=sources_text,
        answer=result["answer"],
    )

    response = client.chat.completions.create(
        model=CONFIG["llm_model"],
        temperature=0.0,
        max_tokens=300,
        messages=[{"role": "user", "content": prompt}],
    )

    try:
        text = response.choices[0].message.content
        match = re.search(r'\{[^}]+\}', text, re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception:
        pass
    return {"error": "Failed to parse evaluation"}


def run_evaluation(results: List[dict]):
    """Evaluate a batch of results and print summary."""
    print("Running evaluation...\n")
    evals = []
    for i, r in enumerate(results):
        print(f"  Evaluating {i+1}/{len(results)}: {r['query'][:50]}...")
        ev = evaluate_result(r)
        ev["query"] = r["query"]
        evals.append(ev)

    valid = [e for e in evals if "error" not in e]
    if valid:
        avg_f = np.mean([e["faithfulness"] for e in valid])
        avg_r = np.mean([e["relevance"] for e in valid])
        avg_c = np.mean([e["citation_quality"] for e in valid])

        print(f"\n{'='*50}")
        print(f"EVALUATION SUMMARY ({len(valid)} questions)")
        print(f"{'='*50}")
        print(f"  Faithfulness     : {avg_f:.1f} / 5")
        print(f"  Relevance        : {avg_r:.1f} / 5")
        print(f"  Citation Quality : {avg_c:.1f} / 5")
        print(f"  Overall Average  : {(avg_f + avg_r + avg_c) / 3:.1f} / 5")

        print(f"\nPer-question:")
        for e in evals:
            q = e.get("query", "?")[:50]
            if "error" not in e:
                print(f"  {q}...")
                print(f"    F={e['faithfulness']} R={e['relevance']} C={e['citation_quality']}")
                print(f"    {e.get('reasoning', '')[:80]}")
            else:
                print(f"  {q}... → Error")
    return evals

print("Evaluation functions defined.")

Evaluation functions defined.


In [33]:
# ── Run Evaluation ────────────────────────────────────────
if 'results' in dir() and results:
    evals = run_evaluation(results)
else:
    print("⚠️  Run the Q&A pipeline first to get results.")

Running evaluation...

  Evaluating 1/2: What is the FY2018 capital expenditure amount (in ...
  Evaluating 2/2: What is the correct religion?...

EVALUATION SUMMARY (2 questions)
  Faithfulness     : 5.0 / 5
  Relevance        : 5.0 / 5
  Citation Quality : 3.0 / 5
  Overall Average  : 4.3 / 5

Per-question:
  What is the FY2018 capital expenditure amount (in ...
    F=5 R=5 C=5
    The answer is fully supported by the source, specifically Source 4, which provid
  What is the correct religion?...
    F=5 R=5 C=1
    The answer accurately reflects the lack of relevant information in the provided 


---
## 9. Interactive Q&A

Uncomment the last line to start an interactive session.
Type questions and get grounded answers. Type `quit` to exit.

In [ ]:
def interactive_qa():
    print("╔════════════════════════════════════════════╗")
    print("║   Document Q&A — Interactive Mode          ║")
    print("║   Type 'quit' to exit                      ║")
    print("╚════════════════════════════════════════════╝")

    while True:
        print()
        q = input("❓ Question: ").strip()
        if q.lower() in ("quit", "exit", "q"):
            print("👋 Done!")
            break
        if not q:
            continue
        ask(q, index, CONFIG)

# Uncomment to start:
# interactive_qa()

---
## 10. Design Notes

### Architecture
```
PDF → PyMuPDF Parser → Section-Aware Chunker → [FAISS + BM25] → RRF Fusion → LLM + Citations
```

### Key Decisions

| Decision | Choice | Why |
|----------|--------|-----|
| PDF Parser | PyMuPDF | Fast, structure-aware, table support |
| Chunking | Heading-grouped + token overlap | Preserves topical coherence |
| Dense Index | FAISS flat + bge-base-en-v1.5 | Strong retrieval, manageable size |
| Sparse Index | BM25 | Essential for exact terms, numbers, acronyms |
| Fusion | Reciprocal Rank Fusion | Score-agnostic, no normalization needed |
| LLM | gpt-4o-mini | Cost-effective, strong instruction following |
| Citations | Source numbering + page refs | Clear, verifiable |

### Limitations
1. **Tables** — Simple tables work; complex merged cells may not parse correctly
2. **Images/Charts** — Text-only extraction; figures are ignored
3. **Heading detection** — Font-size heuristic works for most docs, not all
4. **Single document** — Designed for one doc at a time

### Domain Generalization
Nothing here is finance-specific. The chunking is structure-based, embeddings are
general-purpose, and the prompt is domain-agnostic. To specialize: fine-tune
embeddings on domain data, or add domain-specific instructions to the prompt.

### Future Improvements
1. **Cross-encoder re-ranker** — Add a second-stage re-ranker for better precision
2. **Iterative retrieval** — If first retrieval is weak, reformulate and retry
3. **Index caching** — Save FAISS + BM25 to disk to avoid re-embedding
4. **Semantic chunking** — Split at natural topic boundaries using embeddings
5. **Multimodal** — Process charts/images with a vision model
6. **Query decomposition** — Break complex questions into sub-questions